<a href="https://colab.research.google.com/github/ks-chauhan/Parking-Slot-Application/blob/main/Notebooks/Fast_Dataset_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import shutil

src = "/content/drive/MyDrive/Parking Slot Project/Parking Slot Datasets"
dst = "/content/parking_dataset"

shutil.copytree(src, dst)

'/content/parking_dataset'

In [4]:
import os

os.listdir("/content/parking_dataset")

['B', 'A']

In [5]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

source_root = "/content/parking_dataset"
output_root = "/content/processed_dataset"

classes = ["busy", "free"]

for split in ["train", "val"]:
    for cls in classes:
        os.makedirs(
            os.path.join(output_root, split, cls),
            exist_ok=True
        )

all_images = {
    "busy": [],
    "free": []
}

# Collect images
for section in ["A", "B"]:

    for cls in classes:

        folder = os.path.join(
            source_root,
            section,
            cls
        )

        for img in os.listdir(folder):

            img_path = os.path.join(folder, img)

            all_images[cls].append(img_path)

# Split and copy
for cls in classes:

    train_imgs, val_imgs = train_test_split(
        all_images[cls],
        test_size=0.15,
        random_state=42,
        shuffle=True
    )

    for img_path in train_imgs:

        shutil.copy(
            img_path,
            os.path.join(
                output_root,
                "train",
                cls
            )
        )

    for img_path in val_imgs:

        shutil.copy(
            img_path,
            os.path.join(
                output_root,
                "val",
                cls
            )
        )

print("Dataset prepared successfully!")

Dataset prepared successfully!


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import os

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [8]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
train_dataset = datasets.ImageFolder(
    "/content/processed_dataset/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "/content/processed_dataset/val",
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(train_dataset.class_to_idx)

{'busy': 0, 'free': 1}


In [17]:
model = models.resnet18(pretrained=True)

In [18]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

model = model.to(device)

In [19]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [20]:
epochs = 3

train_losses = []
val_accuracies = []

for epoch in range(epochs):

    # ---------------- TRAIN ----------------

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    train_losses.append(epoch_loss)

    # ---------------- VALIDATION ----------------

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    val_accuracies.append(accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Val Accuracy: {accuracy:.2f}%"
    )

Epoch [1/3] Loss: 0.0648 Val Accuracy: 99.15%
Epoch [2/3] Loss: 0.0218 Val Accuracy: 99.79%
Epoch [3/3] Loss: 0.0140 Val Accuracy: 99.74%


In [21]:
torch.save(
    model.state_dict(),
    "/content/parking_resnet18.pth"
)

print("Model saved successfully!")

Model saved successfully!
